# AI-Powered Personal Spending & Financial Behavior Coach
## Notebook 03 — Preprocessing & Training Data Development

### Objective

Prepare a modeling-ready dataset to support:

1. Unsupervised clustering (persona development for the AI coach)
2. Optional supervised classification (high-spender prediction for grading robustness)

This notebook performs:
- Categorical encoding
- Feature scaling
- Target creation (optional supervised task)
- Train/test split
- Dataset persistence for modeling

## 1) Load Customer-Level Behavioral Dataset

We begin with the aggregated customer-level feature table generated
during the wrangling phase.

This dataset contains:
- total_spend
- avg_spend
- spend_std
- transaction_count
- unique_categories
- age
- gender

These features will form the basis for clustering and classification.

In [14]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

customer_df = pd.read_csv("customer_behavior_features.csv")
customer_df.head()

,customer_id,total_spend,avg_spend,spend_std,transaction_count,unique_categories,age,gender
0,29,27.02,27.02,0.0,1,1,42,M
1,51,1898.56,1898.56,0.0,1,1,21,Unknown
2,54,166.30,166.30,0.0,1,1,44,M
3,83,125.85,125.85,0.0,1,1,74,Unknown
4,90,18.16,18.16,0.0,1,1,20,M


The dataset is successfully loaded and contains both numeric and categorical features.

Next step:
Prepare categorical variables for modeling.

## 2) Encode Categorical Variables

The dataset contains categorical data:
- gender

Machine learning models require numeric input.
Therefore, we convert categorical variables into dummy/indicator variables
using one-hot encoding.

We use `drop_first=True` to avoid multicollinearity.

In [18]:
customer_df["gender"] = customer_df["gender"].astype(str)

customer_encoded = pd.get_dummies(
    customer_df,
    columns=["gender"],
    drop_first=True
)

customer_encoded.head()

,customer_id,total_spend,avg_spend,spend_std,transaction_count,unique_categories,age,gender_M,gender_Unknown
0,29,27.02,27.02,0.0,1,1,42,True,False
1,51,1898.56,1898.56,0.0,1,1,21,False,True
2,54,166.30,166.30,0.0,1,1,44,True,False
3,83,125.85,125.85,0.0,1,1,74,False,True
4,90,18.16,18.16,0.0,1,1,20,True,False


The categorical variable `gender` has now been converted into numeric indicator columns.

These features can now be safely included in modeling.

## 3) Prepare Feature Matrix for Clustering

For clustering:
- We exclude any identifier fields (e.g., customer_id if present).
- We retain only modeling features.

Clustering is sensitive to feature magnitude, so scaling will be required next.

In [22]:
if "customer_id" in customer_encoded.columns:
    customer_encoded = customer_encoded.drop(columns=["customer_id"])

X = customer_encoded.copy()
X.head()

,total_spend,avg_spend,spend_std,transaction_count,unique_categories,age,gender_M,gender_Unknown
0,27.02,27.02,0.0,1,1,42,True,False
1,1898.56,1898.56,0.0,1,1,21,False,True
2,166.30,166.30,0.0,1,1,44,True,False
3,125.85,125.85,0.0,1,1,74,False,True
4,18.16,18.16,0.0,1,1,20,True,False


The feature matrix `X` now contains all modeling-ready features.

Next:
Standardize feature magnitudes to ensure fair clustering.

## 4) Standardize Feature Magnitudes

Clustering algorithms (e.g., KMeans) are distance-based.
Features with larger magnitudes would otherwise dominate the model.

We apply StandardScaler:
- Mean = 0
- Standard deviation = 1

This ensures each feature contributes proportionally.

In [26]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled[:5]

array([[-0.65715258, -0.65715258,  0.        ,  0.        ,  0.        ,
        -0.3860618 ,  1.11722936, -0.33507136],
       [ 2.30572287,  2.30572287,  0.        ,  0.        ,  0.        ,
        -1.63375299, -0.89507136,  2.98443887],
       [-0.4366554 , -0.4366554 ,  0.        ,  0.        ,  0.        ,
        -0.26723406,  1.11722936, -0.33507136],
       [-0.50069267, -0.50069267,  0.        ,  0.        ,  0.        ,
         1.51518193, -0.89507136,  2.98443887],
       [-0.67117904, -0.67117904,  0.        ,  0.        ,  0.        ,
        -1.69316686,  1.11722936, -0.33507136]])

The scaled feature matrix is now ready for clustering.

We save this matrix to preserve reproducibility.

In [29]:
np.save("X_scaled.npy", X_scaled)
print("Scaled matrix saved.")

Scaled matrix saved.


## 5) Create Supervised Target (High-Spender Classification)

Although clustering drives the AI coach, the capstone rubric requires
supervised modeling comparison.

We define a binary classification target:
- 1 = High Spender (top 25% total spend)
- 0 = Others

This creates a balanced, meaningful classification problem.

In [32]:
threshold = customer_df["total_spend"].quantile(0.75)

customer_encoded["high_spender"] = (
    customer_df["total_spend"] >= threshold
).astype(int)

customer_encoded["high_spender"].value_counts()

high_spender
0    37500
1    12500
Name: count, dtype: int64

The dataset now includes a binary response variable.

Next:
Split into training and testing sets for supervised modeling.

## 6) Train/Test Split

We split the dataset into:
- 80% training
- 20% testing

Stratified sampling ensures class balance is preserved.

In [36]:
X_supervised = customer_encoded.drop(columns=["high_spender"])
y_supervised = customer_encoded["high_spender"]

X_train, X_test, y_train, y_test = train_test_split(
    X_supervised,
    y_supervised,
    test_size=0.2,
    random_state=42,
    stratify=y_supervised
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (40000, 8)
Test size: (10000, 8)


The supervised dataset is now properly partitioned.

This ensures fair evaluation of classification models in the next notebook.

## 7) Persist Preprocessed Data

To maintain reproducibility and modular workflow:

We save:
- Scaled matrix for clustering
- Train/test splits for classification

In [40]:
X_train.to_csv("X_train.csv", index=False)
X_test.to_csv("X_test.csv", index=False)
y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

print("Preprocessed datasets saved successfully.")

Preprocessed datasets saved successfully.


## Preprocessing Summary

Completed:

✔ Encoded categorical features  
✔ Standardized numeric magnitudes  
✔ Created clustering-ready feature matrix  
✔ Defined supervised classification target  
✔ Performed train/test split  
✔ Saved datasets for modeling  
